# 03. Dual-stream CrossDiT K6: Audio ↔ Video по одному оператору

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Выведем asymmetric Q/K/V, точный порядок pre/post text states и modulation routing; проверим ручную сборку, gradients и directional cuts.

[Маршрут](../README.md) · [Точные задания MIT](../MIT_BRIDGE.md) · [Методы и source map](../SOURCE_MAP.md) · [Измерения](../reports/VALIDATION.md).

Каждый опыт: **теория → формула → формы/единицы → прогноз → код → измерение → ограниченный вывод**. CPU по умолчанию; без pretrained weights, сети и ComfyUI jobs.


**Связь с MIT:** Предпосылки MIT: notes §6.1 / Lab 3 DiT. Асимметричные Audio/Video streams, fusion gates и раздельные time embeddings — дополнение K6, отсутствующее в исходном MNIST Lab 3.

[MIT course](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf).


In [ ]:
from pathlib import Path
import sys,math,json,copy
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
COURSE=next((p/'dit' for p in [Path.cwd(),*Path.cwd().parents] if (p/'dit/labkit/core.py').is_file()),None)
assert COURSE is not None,'Clone the study repository first; see dit/README.md (Colab setup).'
sys.path.insert(0,str(COURSE))
from labkit.core import (native,tiny_model,tiny_inputs,activate_gates,fresh_forward,
  load_k5_block,manual_single,manual_fused,CountingLinearField,tiny_bundle,
  clone_bundle,scalar_bundle,vector_from_bundle,dummy_text,show_source,
  check_sources,save_metrics,figure_path)
torch.set_num_threads(2);torch.manual_seed(42)
LAB='03_dual_stream_crossdit_k6';metrics={}
print('Pinned source files checked:',check_sources())


## 1. Два генерируемых состояния и разные widths

$$
V\in\mathbb{R}^{B\times N_v\times D_v},\qquad
A\in\mathbb{R}^{B\times N_a\times D_a}.
$$

Released Lite config: D_v=1792, D_a=896, head dimension 64: h_v=28,h_a=14. Tiny test ниже: D_v=48,D_a=32,d=8. Video и audio имеют собственные self-attention, text refiners, time embeddings, FFN и output heads.

Для **обновления video из audio**:

$$
Q_v=V_qW_Q^{v},\quad K_{a\to v}=A_{pre}W_K^{a\to v},\quad
V_{a\to v}=A_{pre}W_V^{a\to v}.
$$

W_Q: D_v→D_v, W_K/W_V: D_a→D_v. Attention имеет B,h_v,N_v,N_a; результат снова B,N_v,D_v. Обратная ветвь проецирует D_v→D_a и сохраняет N_a. Не требуются одинаковые sequence lengths или hidden widths. Число parameters одной video-receiving CA: Q/O дают 2D_v², K/V — 2D_aD_v; четыре biases дают 4D_v; Q/K RMSNorm — 2d_v. В обеих directions:

$$
P_{cross}=2(D_v+D_a)^2+4(D_v+D_a)+2d_v+2d_a.
$$

Это только два CA modules: внешние modulation/norm и self-attention не включены. Имена source va/av не используем как доказательство направления: направление задаёт **получатель Q**.


In [ ]:
model=tiny_model();block=model.visual_transformer_blocks[0];activate_gates(block)
Dv,Da=48,32
v=torch.randn(1,8,Dv);a=torch.randn(1,6,Da)
tv=torch.randn(1,5,Dv);ta=torch.randn(1,5,Da);cv=torch.randn(1,16);ca=torch.randn(1,16)
rv=model.visual_rope((2,2,2),[torch.arange(2)]*3).flatten(0,2);ra=model.audio_rope(torch.arange(6))
params=sum(p.numel() for m in [block.va_cross_attention,block.av_cross_attention] for p in m.parameters())
expected=2*(Dv+Da)**2+4*(Dv+Da)+2*8+2*8 # biases + two Q/K RMSNorm sets
assert params==expected
metrics['cross_attention_parameters']={'measured':params,'formula':expected,'scope':'two cross-attention modules only, excludes gates/norm outside attention'}
print(metrics['cross_attention_parameters'])


## 2. Порядок внутри FusedTransformerDecoderBlock

Сначала каждый поток делает self-attention. После normalized state перед text CA сохраняются V_pre и A_pre. Text update добавляется в каждый поток. Затем:

$$
Q_v\leftarrow\mathrm{M}_{va}(V_{after\ text}),\qquad
Q_a\leftarrow\mathrm{M}_{av}(A_{after\ text}),
$$

$$
\Delta V=\mathrm{CA}(Q_v,K/V=A_{pre}),\qquad
\Delta A=\mathrm{CA}(Q_a,K/V=V_{pre}).
$$

![Схема двухпоточного блока: Q после text update, K/V до него](../../kandinsky-6/reports/figures/components/dual_stream_architecture.png)

Схема из существующей K6 component lab08; arrows показывают сохранённые pre-CA states, а не уже обновлённые outputs.

Оба cross outputs вычисляются до применения любого из них. Следовательно A→V и V→A используют сохранённый pre-text контекст, а queries — post-text state. Простая схема «сначала обновим video, потом из обновлённого video возьмём audio K/V» была бы другой сетью.

Ручная реализация из math ledger сравнивается с original forward. Затем намеренно подставляем post-text K/V, чтобы проверить, что утверждение о порядке действительно влияет на результат.


In [ ]:
with torch.no_grad():
    original=block(v,a,tv,ta,(cv,ca),rv,ra)
    manual=manual_fused(block,v,a,tv,ta,cv,ca,rv,ra)
    wrong=manual_fused(block,v,a,tv,ta,cv,ca,rv,ra,kv_after_text=True)
parity=max(float((x-y).abs().max()) for x,y in zip(original,manual))
wrong_error=max(float((x-y).abs().max()) for x,y in zip(original,wrong))
assert parity<2e-6 and wrong_error>1e-5
metrics['manual_block_max_error']=parity;metrics['wrong_KV_order_max_error']=wrong_error
show_source('core/components/dit.py','FusedTransformerDecoderBlock')


## 3. Cross gates и два времени

Shift/scale/gate приходят из va/av-modulation. В released flags fix_modulation=True, cross_gates=True:

- video query shift/scale зависит от c_v, а gate обновления video — от c_a;
- audio query shift/scale зависит от c_a, а gate обновления audio — от c_v.

Причина — не название flag, а конкретный split: va output содержит D_v+D_v+D_a, av output — D_a+D_a+D_v. Gate последней части уходит потоку соответствующей ширины. При других комбинациях flags routing меняется. Код требует согласованных time dimensions; released оба равны 512, в tiny оба 16.

Сравним original/manual при всех четырёх flags. Затем отключим только одну cross residual связь: audio perturbation больше не должна менять video внутри этого блока. В полном stack косвенные пути через следующие blocks надо анализировать отдельно.


In [ ]:
routing=[]
for fix in [False,True]:
    for cross in [False,True]:
        b=native().FusedTransformerDecoderBlock(48,16,96,8,32,16,64,8,engine='sdpa',ca_rope=True,cross_gates=cross,fix_modulation=fix)
        activate_gates(b)
        with torch.no_grad():
            actual=b(v,a,tv,ta,(cv,ca),rv,ra);manual=manual_fused(b,v,a,tv,ta,cv,ca,rv,ra)
        err=max(float((x-y).abs().max()) for x,y in zip(actual,manual));assert err<2e-6
        routing.append({'fix_modulation':fix,'cross_gates':cross,'manual_error':err})
metrics['flag_parity']=routing
with torch.no_grad():
    base=block(v,a,tv,ta,(cv,ca),rv,ra)
    a_changed=a+.3*torch.randn_like(a)
    perturbed=block(v,a_changed,tv,ta,(cv,ca),rv,ra)
    cut=block(v,a,tv,ta,(cv,ca),rv,ra,va_gate_scale=0.)
    cut_perturbed=block(v,a_changed,tv,ta,(cv,ca),rv,ra,va_gate_scale=0.)
assert torch.equal(cut[0],cut_perturbed[0])
metrics['audio_to_video_open_MSE']=float((base[0]-perturbed[0]).square().mean())
metrics['audio_to_video_cut_error']=float((cut[0]-cut_perturbed[0]).abs().max())
assert metrics['audio_to_video_open_MSE']>0


## 4. Измеряем cross dependence градиентом

Скалярная функция выбранного video output имеет Jacobian по audio input. При открытой ветви norm градиента должен быть ненулевым; при cut — нулевым/unused. Это проверка вычислительной связи, а не доказательство lip-sync, семантической согласованности или качества audio.

Отдельная модальность может присутствовать в input, но не обновляться solver. Это conditioning на clean stream. Удаление её tensor — другая ветка forward, а обнуление cross gate — ещё другой intervention.


In [ ]:
a_probe=a.clone().requires_grad_(True)
y=block(v,a_probe,tv,ta,(cv,ca),rv,ra)[0]
g=torch.autograd.grad(y.square().mean(),a_probe)[0]
ycut=block(v,a_probe,tv,ta,(cv,ca),rv,ra,va_gate_scale=0.)[0]
gcut=torch.autograd.grad(ycut.square().mean(),a_probe,allow_unused=True)[0]
metrics['cross_jacobian_norm']=float(g.norm());metrics['cut_jacobian_norm']=0. if gcut is None else float(gcut.norm())
assert metrics['cross_jacobian_norm']>0 and metrics['cut_jacobian_norm']==0
fig,axes=plt.subplots(1,2,figsize=(10,3));axes[0].bar(['Correct manual','Wrong K/V order'],[parity,wrong_error]);axes[0].set(ylabel='Output max difference',title='Ordering control');axes[1].bar(['A→V open','A→V cut'],[metrics['cross_jacobian_norm'],metrics['cut_jacobian_norm']]);axes[1].set(ylabel='Audio→video Jacobian norm',title='Directional dependence')
fig.tight_layout();fig.savefig(figure_path(LAB,'fusion_controls'));plt.show()


## Измерение → вывод

Ручная сборка совпала с native fused block при всех flags; неверный pre/post K/V порядок дал расхождение. Directional cut устранил выбранную cross зависимость. Проверены function/gradient contracts, не качество синхронизации.


## Самостоятельные опыты

1. Повторите directional cut для V→A (`av_gate_scale=0`).
2. Найдите source inputs gates для всех flags без подглядывания в результат.
3. Сравните h_v N_v N_a + h_a N_a N_v с joint SA на N_v+N_a.
4. Объясните, почему random tiny Jacobian не доказывает trained AV synchronization.

Перед изменением запишите прогноз. Сохраняйте baseline, меняйте один фактор, отмечайте seed и tensor layout. Нулевая ошибка на oracle не равна pretrained quality.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Первичные источники

[MIT course / notes](https://diffusion.csail.mit.edu/2026/index.html) · [DiT](https://arxiv.org/abs/2212.09748) · [Flow Matching](https://arxiv.org/abs/2210.02747) · [K5](https://arxiv.org/abs/2511.14993) · [K6 §3.2, §4.5](https://arxiv.org/html/2610.05608v1) · [π-Flow §3.1, §4 / Alg.1](https://arxiv.org/html/2510.14974v1) · [Source hashes / provenance](../sources.json).

Формулы ниже/выше выводятся для явно заданных соглашений. Тела source methods сохраняются; adapters и doubles подписаны. Малые experiments не воспроизводят промышленное обучение.
